# Exploration : `archive_rate` et `conformity_bias` vs données empiriques

**Objectif.** Comprendre quel(s) paramètre(s) du modèle Wright-Fisher à archivage cumulatif (`code/simulation.py`) rapprochent la distribution simulée des catégories (fandom) de la distribution empirique réelle (`data/fandom_data.csv`, 176 catégories).

**Métriques de référence** (calculées une fois sur les 176 catégories empiriques, réutilisées comme lignes pointillées rouges dans tous les graphes ci-dessous) :

| Métrique | Valeur empirique | Sens |
|---|---|---|
| `gini` | 0.796 | 0 = parfaitement égal, 1 = totalement inégal |
| `hill_0` | 176 | richesse (nb de catégories représentées) |
| `hill_1` | 45.51 | diversité effective (exp(Shannon)) |
| `hill_2` | 26.54 | diversité effective (inverse de Simpson, pénalise les catégories rares) |
| `hill_inf` | 9.78 | inverse de Berger-Parker (domination de la catégorie la plus fréquente) |

**Déroulé.** On part d'une question simple — *"est-ce que `archive_rate` rapproche le modèle de l'empirique ?"* — et on suit les résultats de section en section : chaque section répond à la question posée par la précédente.

Tous les scripts sont exécutés avec l'environnement dédié du projet :
```bash
PY="/Users/rolly/Documents/10-19_Université_et_scolarité/PhD/phd-env/bin/python"
```

---
## 1. Distributions rang-fréquence selon `archive_rate`, avec ajustement MLE

**Script** : `code/plot_archive_rate_mle.py` (nouveau — réimplémente l'estimateur MLE de loi de puissance discrète à la main, méthode Clauset–Shalizi–Newman 2009, pour éviter de réintroduire la dépendance `powerlaw` que le projet avait volontairement abandonnée).

Pour chaque valeur de `archive_rate` (0.01 à 0.6), on lance 5 simulations (n=176, résolution empirique), on moyenne leurs distributions de fréquences, on trace le nuage rang-fréquence en log-log, et on ajuste une loi de puissance MLE sur la queue (droite pointillée). La courbe empirique (losanges noirs) est superposée pour référence.

```bash
$PY code/plot_archive_rate_mle.py -N 176 -ni 4400 -nf 8800 \
    --archive_rates 0.01 0.02 0.05 0.1 0.15 0.2 0.3 0.4 0.6 --n_repeats 5
```

![Rang-fréquence et ajustements MLE selon archive_rate](notebook_figures/archive_rate_mle_comparison.png)

**Lecture.** L'exposant MLE simulé reste quasi constant ($\hat\alpha \approx 1.35$–$1.40$) quelle que soit la valeur de `archive_rate` testée — les 9 courbes sont presque parallèles. L'empirique, lui, est plus raide ($\hat\alpha \approx 1.53 \pm 0.05$) : les catégories réelles sont un peu plus concentrées sur la tête de distribution que ce que le modèle produit, et ce quel que soit le taux d'archivage.

→ **`archive_rate` ne semble pas être le levier qui contrôle la *forme* (l'inégalité) de la distribution.** Reste à vérifier si c'est aussi vrai sur les métriques de diversité usuelles (gini, nombres de Hill), pas seulement sur l'exposant de la queue.

---
## 2. Métriques de diversité vs `archive_rate`

**Script** : `code/simulation.py --sweep_param archive_rate` (fonctionnalité déjà existante, `plot_sweep_metrics`).

Sweep sur les mêmes valeurs de `archive_rate`, 5 répétitions par valeur (barres d'erreur = écart-type entre répétitions), à deux résolutions :
- **n=40** : configuration canonique du papier ;
- **n=176** : résolution empirique (comparaison équitable, notamment pour `hill_0` qui est plafonné par `n_classes`).

```bash
$PY code/simulation.py --sweep_param archive_rate \
    --sweep_values 0.01 0.02 0.05 0.1 0.15 0.2 0.3 0.4 0.6 \
    --sweep_repeats 5 -N 40 -ni 1000 -nf 2000 \
    -o outputs/ --plot article/illustration/metrics_vs_archive_rate_n40.png
```

![Métriques vs archive_rate, n=40](notebook_figures/metrics_vs_archive_rate_n40.png)

```bash
$PY code/simulation.py --sweep_param archive_rate \
    --sweep_values 0.01 0.02 0.05 0.1 0.15 0.2 0.3 0.4 0.6 \
    --sweep_repeats 5 -N 176 -ni 4400 -nf 8800 \
    -o outputs/ --plot article/illustration/metrics_vs_archive_rate_n176.png
```

![Métriques vs archive_rate, n=176](notebook_figures/metrics_vs_archive_rate_n176.png)

**Lecture.**
- À **n=40**, `hill_0` ne peut de toute façon pas dépasser 40 (alors que l'empirique en a 176) — comparaison peu informative pour cette métrique à cette résolution.
- À **n=176**, `hill_0` est la seule métrique vraiment sensible à `archive_rate` : elle grimpe de ~125 à ~175 et rejoint presque l'empirique à `archive_rate≈0.4`. Logique : plus on archive vite, plus de catégories rares ont une chance d'entrer dans l'archive cumulative avant qu'elle se sature.
- **`gini`, `hill_1`, `hill_2`, `hill_inf` restent plats sur toute la plage 0.01–0.6**, loin de l'empirique (gini simulé ~0.93–0.94 vs ~0.80 réel ; hill_1/2/inf simulés très en dessous de l'empirique). `archive_rate` ne bouge quasiment pas ces métriques.

→ Confirme la section 1 : **`archive_rate` pilote surtout la richesse (`hill_0`), pas la concentration/l'inégalité de la distribution.** Il faut un autre paramètre pour ça — on teste `conformity_bias`, le paramètre qui contrôle à quel point l'échantillonnage de chaque génération favorise les catégories déjà populaires (exposant sur les comptes : `bins ** conformity_bias`, neutre à 1.0).

---
## 3. Sweep 2D : `archive_rate` × `conformity_bias`

**Scripts** : `code/hpc/build_param_grid.py` (génère la grille) + `code/hpc/run_param_grid.py` (exécute + `plot_grid_heatmap`).

Grille croisée : 5 valeurs de `archive_rate` × 6 valeurs de `conformity_bias` × 5 seeds = 150 runs (n=176).

```bash
$PY code/hpc/build_param_grid.py -N 176 -ni 4400 -nf 8800 \
    --archive_rates 0.01 0.05 0.1 0.2 0.4 \
    --conformity_biases 0.5 0.7 0.85 1.0 1.15 1.3 \
    --n_seeds 5 -o params_2d.csv

$PY code/hpc/run_param_grid.py params_2d.csv -o results_2d.csv --n_jobs 4 \
    --vary_param archive_rate --vary_param2 conformity_bias \
    --plot article/illustration/heatmap_archive_conformity.png
```

![Heatmap archive_rate x conformity_bias](notebook_figures/heatmap_archive_conformity.png)

**Lecture.** Sur chaque panneau, les bandes horizontales dominent largement les variations verticales : **`conformity_bias` a un effet énorme, `archive_rate` un effet négligeable en comparaison.**

Il y a une **transition très abrupte autour de `conformity_bias=1.0`** :

| `conformity_bias` | hill_1 (sim) | hill_2 (sim) | hill_inf (sim) | gini (sim) |
|---|---|---|---|---|
| 0.85 | ~169 | ~168 | ~150 | ~0.07 |
| **1.00** | ~16 | ~11 | ~6 | ~0.93 |
| 1.15 | ~1.0 | ~1.0 | ~1.0 | ~0.994 |

En dessous de `conformity_bias≈0.9`, le modèle sur-diversifie (quasi-uniforme sur les 176 catégories). Au-dessus de `conformity_bias≈1.0`, il s'effondre sur une poignée de catégories dominantes. **La cible empirique (gini=0.80, hill_1=45.5) tombe quelque part entre 0.85 et 1.00** — pas à un point testé ici. On resserre la grille sur cette fenêtre.

---
## 4. Sweep fin de `conformity_bias` (0.84–1.00)

`archive_rate` fixé à 0.1 (effet négligeable, cf. section 3). 13 valeurs de `conformity_bias`, 8 seeds chacune.

```bash
$PY code/hpc/build_param_grid.py -N 176 -ni 4400 -nf 8800 \
    --archive_rates 0.1 \
    --conformity_biases 0.84 0.86 0.88 0.90 0.91 0.92 0.93 0.94 0.95 0.96 0.97 0.98 1.00 \
    --n_seeds 8 -o params_conformity_fine.csv

$PY code/hpc/run_param_grid.py params_conformity_fine.csv -o results_conformity_fine.csv --n_jobs 4 \
    --vary_param conformity_bias \
    --plot article/illustration/metrics_vs_conformity_bias_fine.png
```

![Métriques vs conformity_bias, 0.84-1.00](notebook_figures/metrics_vs_conformity_bias_fine.png)

**Lecture.** Résultat net : **les 4 métriques (`gini`, `hill_1`, `hill_2`, `hill_inf`) croisent leur ligne empirique dans une fenêtre étroite, 0.988–0.998** (croisements calculés par interpolation linéaire entre points adjacents). C'est un signal fort et cohérent : plusieurs métriques indépendantes convergent vers *le même* petit intervalle de `conformity_bias`.

`hill_0` (richesse), lui, reste sous l'empirique sur toute la fenêtre (~160–165 vs 176) — cohérent avec la section 2 : c'est `archive_rate`, pas `conformity_bias`, qui le rapprocherait de la cible.

→ On zoome une dernière fois sur 0.98–1.00 pour affiner le point de croisement.

---
## 5. Zoom final (0.98–1.00)

9 valeurs entre 0.980 et 1.000, 12 seeds chacune (davantage de répétitions pour resserrer les barres d'erreur sur un intervalle aussi étroit).

```bash
$PY code/hpc/build_param_grid.py -N 176 -ni 4400 -nf 8800 \
    --archive_rates 0.1 \
    --conformity_biases 0.980 0.984 0.988 0.990 0.992 0.994 0.996 0.998 1.000 \
    --n_seeds 12 -o params_conformity_zoom.csv

$PY code/hpc/run_param_grid.py params_conformity_zoom.csv -o results_conformity_zoom.csv --n_jobs 4 \
    --vary_param conformity_bias \
    --plot article/illustration/metrics_vs_conformity_bias_zoom.png
```

![Métriques vs conformity_bias, zoom 0.98-1.00](notebook_figures/metrics_vs_conformity_bias_zoom.png)

**Lecture — points de croisement précis (interpolation linéaire) :**

| Métrique | `conformity_bias` au croisement | Valeur simulée au point le plus proche testé |
|---|---|---|
| `gini` | ≈ 0.990 | 0.793 (à 0.990) vs 0.796 empirique |
| `hill_1` | ≈ 0.990 | 45.9 (à 0.990) vs 45.5 empirique |
| `hill_2` | ≈ 0.996 | 25.2 (à 0.996) vs 26.5 empirique |
| `hill_inf` | ≈ 0.998 | 11.1 (à 0.998) vs 9.8 empirique |

`gini` et `hill_1` s'alignent quasi parfaitement à **`conformity_bias≈0.990`**. `hill_2` et `hill_inf` demandent une valeur très légèrement plus élevée (~0.996–0.998). Les 4 métriques ne convergent donc pas exactement au même point — il y a une petite tension résiduelle entre elles — mais toutes se resserrent dans une fenêtre de moins de 0.01 en largeur, ce qui est remarquablement étroit compte tenu de l'ampleur de la transition observée en section 3.

---
## 6. Sweep combiné final : `archive_rate` (large) × `conformity_bias` (affiné)

Suite logique de la section 5 : on croise maintenant `archive_rate ∈ [0.2, 0.6]` (la plage où `hill_0` est déjà proche de l'empirique, cf. section 2) avec `conformity_bias ∈ [0.988, 1.000]` (la fenêtre de transition affinée en section 5), pour chercher le point qui minimise l'écart sur les **5 métriques à la fois** — pas une à la fois comme avant. 5 valeurs × 7 valeurs × 10 seeds = 350 runs (n=176).

```bash
$PY code/hpc/build_param_grid.py -N 176 -ni 4400 -nf 8800 \
    --archive_rates 0.2 0.3 0.4 0.5 0.6 \
    --conformity_biases 0.988 0.990 0.992 0.994 0.996 0.998 1.000 \
    --n_seeds 10 -o params_combined.csv

$PY code/hpc/run_param_grid.py params_combined.csv -o results_combined.csv --n_jobs 4 \
    --vary_param archive_rate --vary_param2 conformity_bias \
    --plot article/illustration/heatmap_combined_final.png
```

![Heatmap final archive_rate x conformity_bias affiné](notebook_figures/heatmap_combined_final.png)

**Lecture.** Les panneaux confirment sans ambiguïté la section 3 : les bandes sont quasiment horizontales — `conformity_bias` domine, `archive_rate` n'a plus qu'un effet mineur (surtout visible sur `hill_0`, qui reste stable ~170 sur toute la plage 0.2–0.6, proche de l'empirique 176).

Moyennées sur `archive_rate` (0.2–0.6), les métriques croisent l'empirique à :

| Métrique | Croisement `conformity_bias` |
|---|---|
| `gini` | ≈ 0.9905 |
| `hill_1` | ≈ 0.9903 |
| `hill_2` | ≈ 0.9954 |
| `hill_inf` | ≈ 0.9980 |

— cohérent avec la section 5 (~0.990 / ~0.996 / ~0.998), maintenant confirmé sur une plage d'`archive_rate` plus large et avec plus de répétitions (10 seeds).

**Meilleur compromis global** (minimise la somme des écarts relatifs sur les 5 métriques à la fois) : **`archive_rate=0.3`, `conformity_bias=0.996`** —

| | gini | hill_0 | hill_1 | hill_2 | hill_inf |
|---|---|---|---|---|---|
| simulé | 0.865 | 169.7 | 31.1 | 25.4 | 14.4 |
| empirique | 0.796 | 176 | 45.5 | 26.5 | 9.8 |

`hill_0` et `hill_2` sont très proches de la cible ; `gini` est raisonnable (+9%) ; `hill_1` et `hill_inf` restent les plus éloignés (le premier sous-estimé, le second sur-estimé) — la tension entre métriques identifiée en section 5 n'est pas totalement résorbable par un seul point de `(archive_rate, conformity_bias)` : aucune combinaison testée ne fait mieux que ~0.96 de somme d'écarts relatifs sur les 5 métriques.

---
## 7. Ajout d'un paramètre `distribution` à `simulation.py`

Jusqu'ici, la distribution initiale de la population (à t=0) était figée en dur dans `run_simulation` : proportions en loi de puissance (rang⁻¹) par défaut, ou proportions empiriques imposées (mode `--validate`). Pour pouvoir tester l'effet de cette forme initiale indépendamment des autres paramètres, une nouvelle fonction `initial_distribution(n_classes, distribution, rng, custom_probs=None)` a été ajoutée à `code/simulation.py`, avec 4 choix :

| `distribution` | Comportement |
|---|---|
| `power_law` (défaut) | rang⁻¹, comportement historique inchangé |
| `uniform` | toutes les classes équiprobables |
| `random` | composition aléatoire (tirage Dirichlet(1,...,1)), reproductible via `--seed` |
| `custom` | proportions fournies via `--distribution_path` (CSV, une valeur par ligne, n_classes lignes) |

Le paramètre est propagé dans toute la chaîne d'appel : `run_simulation` → `run_canonical` / `run_multiple_simulations` / `sweep_parameter` → CLI (`main()`). Le mode `--validate` continue d'utiliser les proportions empiriques réelles, indépendamment de `--distribution` (comportement inchangé).

**Tests de validation effectués** (n_classes=10, initial_pop=final_pop=200, generations=20, seed=1, sauf mention contraire) :
- `--distribution power_law` / `uniform` / `random` : trois runs donnant des métriques distinctes et cohérentes (gini le plus bas en `uniform` ≈0.26, le plus élevé et variable en `random` ≈0.60)
- `--distribution custom --distribution_path <csv>` : chargement d'un CSV de 10 proportions, run correct
- Gestion d'erreur : `custom` sans `--distribution_path` → message d'erreur propre (pas de traceback) ; CSV de mauvaise longueur (10 valeurs pour n_classes=5) → `ValueError` explicite
- Propagation vérifiée en mode `--sweep_param archive_rate --distribution uniform` et en mode `--n_runs 2 --distribution random`
- **Non-régression** : run par défaut (`-N 40 -alpha 0.1 -s 7 --validate`, donc `distribution=power_law` implicite) redonne exactement les mêmes métriques qu'avant la modification (gini=0.8991 pour le run canonique n=40).

---
## 8. Extension de `build_param_grid.py` / `run_param_grid.py` pour `--distribution`

`simulation.py --sweep_param` ne fait varier qu'UN paramètre à la fois : pour croiser `archive_rate` × `conformity_bias` avec une distribution initiale donnée, il faut passer par le pipeline de grille (`code/hpc/build_param_grid.py` + `run_param_grid.py`), qui ne connaissait pas encore `distribution`. Deux modifications :

- **`build_param_grid.py`** : nouveaux arguments `--distribution` (choix identiques à `simulation.py`) et `--distribution_path`, ajoutés comme colonnes **constantes** de la grille (répétées sur chaque ligne — ce ne sont pas des axes qu'on balaye, juste une configuration fixée pour tout le sweep).
- **`run_param_grid.py`** : `_run_one_row` lit désormais ces colonnes (`.get()` avec repli sur `power_law` pour rester compatible avec un `params.csv` généré avant cet ajout) et les transmet à `run_simulation`.

**Test de validation** : grille jouet (N=20, ni=nf=200, T=20, `archive_rates=[0.05, 0.2]`, `conformity_biases=[0.9, 1.0]`, `n_seeds=2`, `--distribution uniform` → 8 jobs) exécutée avec succès de bout en bout (build + run + heatmap). Non-régression vérifiée avec une grille par défaut (sans `--distribution`).

---
## 9. Exécution sur serveur distant (`les-lillas`) et sweeps élargis

Pour pousser les plages de `archive_rate`/`conformity_bias` plus loin sans attendre en local, le repo a été synchronisé vers un serveur distant (`les-lillas`, alias SSH déjà configuré côté utilisateur) :

```bash
rsync -avz --exclude='.git' --exclude='__pycache__' --exclude='.DS_Store' --exclude='wf_simulation' \
    ./ les-lillas:~/CultureLab_article_repo/
```

(`wf_simulation` = venv Python déjà présent sur le serveur, exclu du transfert ; transfert non destructif, `--delete` volontairement omis pour ne rien supprimer d'existant côté serveur). Le venv distant avait déjà numpy/pandas/matplotlib/tqdm ; `scipy` a été installé en plus (`pip install scipy`) pour couvrir `plot_archive_rate_mle.py`.

**Sweep A — 960 jobs** (distribution uniforme, n=176, `ni=4400`, `nf=8800`) :
```bash
$PY code/hpc/build_param_grid.py -N 176 -ni 4400 -nf 8800 \
    --archive_rates 0.01 0.05 0.1 0.15 0.2 0.3 0.4 0.5 0.6 0.8 \
    --conformity_biases 0.7 0.75 0.8 0.85 0.9 0.95 1.0 1.05 1.1 1.15 1.2 1.3 \
    --n_seeds 8 --distribution uniform -o params_uniform_wide.csv
$PY code/hpc/run_param_grid.py params_uniform_wide.csv -o results_uniform_wide.csv --n_jobs 8 \
    --vary_param archive_rate --vary_param2 conformity_bias \
    --plot article/illustration/heatmap_uniform_wide.png
```

**Sweep B — 1664 jobs** (plage encore élargie : `archive_rate` jusqu'à 1.0, `conformity_bias` de 0.5 à 1.5) :
```bash
$PY code/hpc/build_param_grid.py -N 176 -ni 4400 -nf 8800 \
    --archive_rates 0.01 0.05 0.1 0.15 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.0 \
    --conformity_biases 0.5 0.6 0.7 0.75 0.8 0.85 0.9 0.95 1.0 1.05 1.1 1.15 1.2 1.3 1.4 1.5 \
    --n_seeds 8 --distribution uniform -o params_uniform_wider.csv
$PY code/hpc/run_param_grid.py params_uniform_wider.csv -o results_uniform_wider.csv --n_jobs 8 \
    --vary_param archive_rate --vary_param2 conformity_bias \
    --plot article/illustration/heatmap_uniform_wider.png
```

(les deux exécutés via `ssh les-lillas '... $PY ...'`, résultats rapatriés en local par `rsync -avz les-lillas:~/CultureLab_article_repo/<fichier> <destination>`)

![Heatmap élargi (960 jobs), distribution uniforme](notebook_figures/heatmap_uniform_wide.png)

![Heatmap très élargi (1664 jobs), distribution uniforme](notebook_figures/heatmap_uniform_wider.png)

**Lecture.** Avec une distribution initiale **uniforme**, `hill_0` sature quasiment partout (~170–175, proche de l'empirique 176) sauf dans un coin `conformity_bias>1.2` + `archive_rate` faible — cohérent avec le fait qu'on démarre déjà réparti sur toutes les classes. La transition gini/hill_1/hill_2/hill_inf autour de `conformity_bias≈1.0` est confirmée, nette, **sur toute la plage élargie** (`conformity_bias` 0.5→1.5, `archive_rate` 0.01→1.0) : aucune deuxième transition n'apparaît ailleurs dans l'espace des paramètres testé. La fenêtre pertinente (celle qui contient l'empirique) reste confinée à `conformity_bias∈[0.9, 1.05]`, quelle que soit la largeur de la plage balayée autour.

---
## 10. Test d'hypothèse : « l'archivage joue-t-il un rôle similaire au biais de conformité ? »

**Script** : `code/test_archive_vs_conformity_hypothesis.py` (nouveau).

**Argument mécaniste de départ.** Dans `run_simulation`, l'archive ne réinjecte jamais rien dans la dynamique générationnelle : chaque génération est entièrement régénérée par `conform_probs` (dépend de `conformity_bias`), et l'archivage se contente d'enregistrer un sous-échantillon **sans remise** de la génération courante, sans influencer la suivante. `archive_rate` devrait donc agir comme une simple **profondeur d'échantillonnage** (comme la profondeur de séquençage en écologie), alors que `conformity_bias` agit **dans** la boucle de reproduction (un mécanisme de sélection/dérive).

**Méthode : raréfaction.** Si `archive_rate` n'est qu'un effet de taille d'échantillon, sous-échantillonner (« raréfier ») toutes les archives à une taille commune avant de calculer les métriques de diversité devrait faire disparaître son effet. L'effet de `conformity_bias`, lui, devrait survivre puisqu'il change la distribution sous-jacente elle-même.

**Paramètres fixes du modèle** (tous les sweeps de cette section) : `n_classes=176`, `initial_pop=4400`, `final_pop=8800`, `generations=1000`, `distribution=uniform`.

**Sweep 1 — `archive_rate`** (isole l'effet pur d'archive_rate) :
- Valeurs : `0.02, 0.05, 0.1, 0.2, 0.4, 1.0` (1.0 = aucune perte, toute génération entièrement enregistrée)
- `conformity_bias` fixé à **1.0** (neutre)
- **8 répétitions** par valeur (seeds 1000–1007)
- Tailles d'archive obtenues : 131 460 → 6 597 400 individus (proportionnelles à `archive_rate`, comme attendu)
- Raréfaction : toutes les archives ramenées à **131 460** individus (la plus petite, celle d'`archive_rate=0.02`), **30 tirages de raréfaction** moyennés par run (seeds 5000+)

**Sweep 2 — `conformity_bias` à `archive_rate=0.1`** (sweep de contrôle, pas de confusion de taille d'échantillon puisque toutes les archives font déjà 659 300 individus) :
- Valeurs : `0.9, 0.95, 1.0, 1.05, 1.1`
- **8 répétitions** (seeds 2000–2007), raréfaction à la même taille commune (quasi no-op), **30 tirages** (seeds 6000+)

**Sweep 3 — `conformity_bias` à `archive_rate=1.0`** (= 100%, archivage sans aucune perte — la condition d'observation la plus favorable possible) :
- Mêmes valeurs de `conformity_bias`, **8 répétitions** (seeds 3000–3007), toutes les archives à 6 597 400 individus, **30 tirages** de raréfaction (seeds 7000+, quasi no-op ici aussi)

**Métriques** : `gini`, `hill_0`, `hill_1`, `hill_2`, `hill_inf` (réutilise `compute_diversity_metrics` de `simulation.py`).

![Raréfaction : archive_rate vs conformity_bias (incl. archivage à 100%)](notebook_figures/rarefaction_archive_vs_conformity.png)

**Lecture — amplitude (max−min) de chaque métrique, brute vs raréfiée :**

| Sweep | Métrique | Amplitude brute | Amplitude raréfiée | Réduction |
|---|---|---|---|---|
| `archive_rate` (0.02→1.0) | `hill_0` | 2.25 | 1.27 | **-43.5%** |
| `archive_rate` (0.02→1.0) | `hill_1` | 1.37 | 1.36 | -0.9% (déjà ~nul) |
| `archive_rate` (0.02→1.0) | `hill_2` | 0.96 | 0.96 | -0.1% (déjà ~nul) |
| `archive_rate` (0.02→1.0) | `hill_inf` | 0.87 | 0.87 | -0.5% (déjà ~nul) |
| `archive_rate` (0.02→1.0) | `gini` | 0.01 | 0.01 | -2.1% (déjà ~nul) |
| `conformity_bias` @ arch=0.1 | `hill_1` | 155.47 | 155.47 | 0.0% |
| `conformity_bias` @ arch=**1.0** | `hill_1` | **155.04** | **155.04** | 0.0% |
| `conformity_bias` @ arch=0.1 | `gini` | 0.83 | 0.83 | 0.0% |
| `conformity_bias` @ arch=**1.0** | `gini` | **0.83** | **0.83** | 0.0% |

**Deux résultats convergents :**

1. **`archive_rate` seul** : son effet sur `hill_0` (le seul non négligeable en brut) chute de 44% après raréfaction à taille commune — la majeure partie de ce qu'on observait était un artefact de taille d'échantillon (courbe de raréfaction classique en écologie). Son effet sur les 4 autres métriques était déjà ~nul en brut et le reste après raréfaction : rien à retirer.

2. **`conformity_bias` à `archive_rate=0.1` vs `archive_rate=1.0`** (comparaison directe, colonne par colonne) : les deux courbes sont **quasi identiques au bruit près** (écarts <1% partout, `hill_0` exactement 176.0 dans les deux cas). Même en observant la population **sans aucune perte d'information** (100% archivé), `conformity_bias` continue de faire chuter `hill_1` de ~156 à ~1.2 et grimper `gini` de 0.16 à 0.99 entre 0.9 et 1.1. Son effet ne dépend donc **pas du tout** du taux d'archivage.

**Verdict : hypothèse réfutée.** `archive_rate` et `conformity_bias` n'agissent pas de façon similaire — ce sont deux mécanismes quasi orthogonaux. `archive_rate` est une profondeur d'échantillonnage a posteriori (n'affecte que la détection des catégories rares via un effet de taille finie) ; `conformity_bias` est un biais de sélection intégré à la boucle de reproduction, dont l'effet (amplitude 100 à 150× plus grande que tout ce qu'`archive_rate` peut produire) est totalement indépendant de la profondeur d'observation — testé et confirmé jusqu'au cas limite de l'archivage à 100%.

---
## 11. Confirmation visuelle : MLE avec `archive_rate` allant jusqu'à 100%

Reprise de la figure de la section 1 (rang-fréquence + ajustement MLE), mais avec des taux d'archivage qui montent vite jusqu'à l'archivage complet, pour vérifier visuellement la conclusion de la section 10 (l'exposant ne devrait pas bouger).

```bash
$PY code/plot_archive_rate_mle.py -N 176 -ni 4400 -nf 8800 \
    --archive_rates 0.1 0.3 0.5 0.7 1.0 --n_repeats 5 \
    -o article/illustration/archive_rate_mle_comparison_to1.png
```

![MLE rang-fréquence, archive_rate 0.1 à 1.0](notebook_figures/archive_rate_mle_comparison_to1.png)

**Lecture.** Les 5 courbes ($\alpha_{arch}$ = 0.1, 0.3, 0.5, 0.7, 1.0) sont **parallèles** : exposant MLE stable entre 1.33 et 1.38 (contre 1.53 pour l'empirique), y compris à `archive_rate=1.0` (archivage à 100%, aucune perte). Seule la hauteur des courbes change (plus d'archivage → plus d'individus captés → décalage vertical), pas leur pente. Confirme visuellement, sur la figure la plus directement lisible du notebook, la conclusion quantitative de la section 10 : `archive_rate` ne modifie pas la forme de la distribution, seulement la profondeur d'observation.

---
## Conclusion

1. **`archive_rate`** contrôle surtout la **richesse** (`hill_0`) via un effet de taille d'échantillon (confirmé par raréfaction, section 10) — quasiment aucun effet structurel sur la forme de la distribution (gini, hill_1, hill_2, hill_inf, exposant MLE). Toute valeur dans `[0.2, 1.0]` amène `hill_0` proche de l'empirique.
2. **`conformity_bias`** contrôle la **concentration/l'inégalité** — effet massif, structurel, totalement indépendant du taux d'archivage (identique à 10% ou 100% d'archivage, section 10). Transition abrupte autour de la neutralité (1.0), confirmée robuste sur une plage élargie `[0.5, 1.5]` (section 9). La fenêtre `conformity_bias ∈ [0.990, 0.998]` fait passer chaque métrique par sa cible empirique, mais pas toutes au même point exact.
3. **Hypothèse « archivage ≈ conformité » réfutée (section 10)** : les deux paramètres ne sont pas substituables. `archive_rate` = profondeur d'observation (post-hoc, sans rétroaction sur la dynamique) ; `conformity_bias` = biais de sélection generatif (dans la boucle, amplitude 100-150× plus grande). Le rapprochement du modèle vers l'empirique repose donc presque entièrement sur `conformity_bias`, pas sur `archive_rate`.
4. **Configuration recommandée comme point de départ** : `archive_rate≈0.3` (pour la richesse), `conformity_bias≈0.996` (meilleur compromis testé sur les 5 métriques simultanément, section 6) — même si un résidu subsiste sur `hill_1`/`hill_inf`.
5. Le modèle a aussi été étendu (sections 7-8) avec un paramètre `distribution` (power_law / uniform / random / custom) pour la population initiale, et le pipeline de sweep 2D est maintenant exécutable sur serveur distant (section 9) pour explorer des plages plus larges sans attendre en local.

**Limite restante.** `hill_1`/`hill_inf` gardent un résidu même au meilleur point de `conformity_bias` (section 6) — un autre mécanisme du modèle (distribution initiale, courbe de croissance de la population) est probablement nécessaire pour fermer complètement l'écart, puisqu'on a maintenant écarté `archive_rate` comme levier possible.